# 07 — Retrieval baseline

The repo has no ground truth, so no change to retrieval can be shown to have
helped rather than hurt. This notebook captures a reproducible baseline over the
eight seed queries from `05_retrieval_testing.ipynb`.

**Run it before and after any retrieval change.** It writes
`SCRIPTS/baseline.json`; the second run diffs against the first.

The pipeline was changed substantially — embeddings moved from a local
384-dim MiniLM to `text-embedding-3-large` at 3072 dims, BM25 now searches the
whole corpus instead of re-weighting dense's top 40, and reranking moved from a
local CrossEncoder to Cohere. Every one of those changes needs this harness to
be worth anything.

In [ ]:
import sys, os, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
try:
    from dotenv import load_dotenv
    load_dotenv(Path.cwd().parent / ".env")
except ImportError:
    pass

import puks_rag
from puks_rag import Corpus

assert os.getenv("AZURE_AI_KEY"), "AZURE_AI_KEY is not set."
corpus = Corpus()

SEED_QUERIES = [
    # Operational
    "How do I reverse a GRN?",
    "Reset a mission in Speed",
    "Resend outbound shipment",
    # Schema
    "What is the primary key of REE_DAT?",
    "Show join between REE_DAT and DOS_DAT",
    "List columns of REE_DAT",
    "How does receipt relate to STK_DAT table?",
    # Text / general
    "Explain warehouse picking process",
]
BASELINE_PATH = Path.cwd() / "baseline.json"
print(f"{len(SEED_QUERIES)} seed queries · baseline at {BASELINE_PATH}")

## Capture

Records the source document of each top-5 hit, which retriever found it, and the Cohere relevance score. Sources — not chunk indices — because indices change on every rebuild.

In [ ]:
current = {}

for q in SEED_QUERIES:
    retrieved, confidence = puks_rag.retrieve_context(corpus, q, top_k=5)
    current[q] = {
        "confidence": round(confidence, 4),
        "hits": [
            {
                "source":    r["metadata"].get("source", "?"),
                "chunk_type": r["metadata"].get("chunk_type", "?"),
                "doc_type":  r["doc_type"],
                "relevance": round(r["relevance_score"], 4),
                "found_by":  [l for l, k in (("dense","in_dense"),("bm25","in_bm25"),("exact","in_exact")) if r[k]],
            }
            for r in retrieved
        ],
    }
    flag = "REFUSE" if confidence < puks_rag.CONFIDENCE_THRESHOLD else "      "
    print(f"[{flag}] {confidence:.4f}  {q}")
    for r in current[q]["hits"]:
        print(f"           {r['relevance']:.4f}  {r['source'][:58]:<58} {','.join(r['found_by'])}")
    print()

## Recall attribution

How many top-5 hits would have been unreachable under the old design, where BM25 only re-ranked what dense retrieval had already found?

In [ ]:
bm25_only = [
    (q, h["source"]) for q, res in current.items() for h in res["hits"]
    if "bm25" in h["found_by"] and "dense" not in h["found_by"]
]
total = sum(len(r["hits"]) for r in current.values())

print(f"top-5 hits total          : {total}")
print(f"reachable only via BM25   : {len(bm25_only)}")
print(f"would have been missed    : {len(bm25_only)/total:.1%} of retrieved context\n")
for q, src in bm25_only:
    print(f"  {src[:64]:<64} ← {q}")

## Diff against the stored baseline

In [ ]:
if BASELINE_PATH.exists():
    previous = json.loads(BASELINE_PATH.read_text())
    for q in SEED_QUERIES:
        before = {h["source"] for h in previous.get(q, {}).get("hits", [])}
        after  = {h["source"] for h in current[q]["hits"]}
        c_before = previous.get(q, {}).get("confidence", 0.0)
        c_after  = current[q]["confidence"]
        if before == after and abs(c_before - c_after) < 0.02:
            print(f"=  {q}")
            continue
        print(f"~  {q}   confidence {c_before:.4f} → {c_after:.4f}")
        for s in sorted(after - before): print(f"     + {s}")
        for s in sorted(before - after): print(f"     - {s}")
else:
    print("No baseline stored yet — the next cell writes one.")

## Store

Commit `baseline.json` so the comparison survives across machines and branches.

In [ ]:
BASELINE_PATH.write_text(json.dumps(current, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"wrote {BASELINE_PATH}")

## What this still is not

These eight queries have **no expected answers**. This harness detects *change*,
not *correctness* — it will tell you retrieval moved, not that it improved.

Turning it into a real evaluation set needs, for each query, the source
documents that genuinely should come back. The best source for those is the
resolved support-ticket queue, which is not in this repo (only two tickets are).
That is [§13's open question 7](../README.md#13-where-this-is-going).